# Show Reel — Post-Level Context & Community Vibe (Colab Enterprise)

Headless, scheduled-executor build of `community_vibe_pipeline.py`.

**Steps:** Post-Level Context Enrichment (Gemini 2.5 Flash) → Local NLP
Enrichment (spaCy `it_core_news_lg`) → Community Vibe & Polarization
(Gemini 2.5 Pro) → `enriched_post_vibe_matrix.parquet` on GCS.

Runs top-to-bottom, no interactivity. Auth = Application Default Credentials
(the Colab Enterprise service account). Reads raw CSVs from
`gs://afb_showreel/`, writes the fused matrix to `gs://afb_showreel/enriched/`.

All LLM prompts live in the `prompts.py` cell below (single source of truth).

In [ ]:
# Cell 1 — dependencies (non-builtin only)
!pip install -q google-cloud-aiplatform gcsfs pyarrow spacy emoji pydantic

In [ ]:
# Cell 2 — Italian spaCy model (has word vectors -> needed for cosine similarity)
!python -m spacy download it_core_news_lg

In [ ]:
# Cell 3 — config + logging (hoisted; authoritative for the whole notebook)
import logging
import sys

# --- GCP config (Colab Enterprise / Vertex AI) -----------------------------
GCP_PROJECT_ID = "gen-lang-client-0792749758"
GCP_BUCKET = "afb_showreel"
GCP_LOCATION = "us-central1"

# --- GCS layout ------------------------------------------------------------
RAW_PREFIX        = f"gs://{GCP_BUCKET}"                           # bucket root — tabular data
MULTIMODAL_PREFIX = f"gs://{GCP_BUCKET}/multimodal_dataset_fixed"  # frames + transcripts
OUTPUT_URI        = f"gs://{GCP_BUCKET}/enriched/enriched_post_vibe_matrix.parquet"

# --- Structured, stdout logging the scheduler can capture ------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
logging.getLogger("community_vibe_pipeline").info(
    "Config loaded: project=%s bucket=%s location=%s", GCP_PROJECT_ID, GCP_BUCKET, GCP_LOCATION
)

In [ ]:
# Cell 3b — test-mode toggles  ← set TEST_MODE=True for a quick smoke run
# ──────────────────────────────────────────────────────────────────────────────
# Master switch: flip to False for a full production run.
TEST_MODE = True

# How many rows to keep per platform when TEST_MODE is True.
TEST_N_POSTS    = 3    # posts per platform fed into Phase 1
TEST_N_COMMENTS = 10   # comments per platform fed into Phase 2/3

# Skip GCS entirely and use the bundled synthetic mock frames.
# Useful when you have no bucket access (e.g. local Colab session).
TEST_USE_MOCK = True

# Disable real Vertex AI calls; Phase 1 & 3 use deterministic fallbacks.
# No quota consumed, no billing, instant completion.
TEST_DISABLE_LLM = True

if TEST_MODE:
    logging.getLogger("community_vibe_pipeline").info(
        "TEST MODE active — n_posts=%d  n_comments=%d  use_mock=%s  disable_llm=%s",
        TEST_N_POSTS, TEST_N_COMMENTS, TEST_USE_MOCK, TEST_DISABLE_LLM,
    )

### Prompt definitions — written to `prompts.py` (edit prompts here)

In [ ]:
%%writefile prompts.py
"""Central prompt registry for ALL Vertex AI / Gemini calls in this project.

╔══════════════════════════════════════════════════════════════════════════╗
║  EDIT YOUR LLM PROMPTS HERE — this is the single source of truth.          ║
║  Nothing else in the pipelines hard-codes prompt text; they import from    ║
║  this module. To tune a prompt, change the constant below and re-run.      ║
╚══════════════════════════════════════════════════════════════════════════╝

Each constant is a ``str.format(...)`` template. The ``{placeholders}`` each
template expects are documented next to it. The ``PROMPTS`` dict at the bottom
lists every prompt by name for quick discovery (e.g. ``python -c "import
prompts; print(list(prompts.PROMPTS))"``).

Consumers
---------
    community_vibe_pipeline.py
        POST_CONTEXT_PROMPT      → Phase 1 (text-only post summary, Flash)
        COMMUNITY_VIBE_PROMPT    → Phase 3 (community vibe over comments, Pro)

    instagram_multimodal_pipeline.py
        MULTIMODAL_POST_PROMPT   → Phase 1 (frames + transcript + caption)
        CAG_SYSTEM_INSTRUCTION   → Phase 3 cache system instruction (CAG)
        CAG_COMMENT_CHUNK_PROMPT → Phase 3 per-chunk comment scoring (CAG)
"""

from __future__ import annotations

from typing import Dict

# --------------------------------------------------------------------------- #
# Phase 1 — Post-Level Context Summarizer (TEXT-ONLY, Gemini 2.5 Flash)
#   placeholders: {platform} {post_text} {transcript} {formats} {tones}
# --------------------------------------------------------------------------- #
POST_CONTEXT_PROMPT = (
    "Sei un analista di media digitali. Analizza il seguente post pubblicato "
    "su {platform}. Combina didascalia e trascrizione (se presente) per "
    "dedurre il contesto creativo.\n\n"
    "--- DIDASCALIA / DESCRIZIONE ---\n{post_text}\n\n"
    "--- TRASCRIZIONE (se disponibile) ---\n{transcript}\n\n"
    "Restituisci ESCLUSIVAMENTE un oggetto JSON conforme allo schema, senza "
    "testo aggiuntivo e senza wrapper markdown. Campi:\n"
    "- format_type: uno tra {formats}\n"
    "- primary_topic: argomento principale conciso (max 6 parole)\n"
    "- intended_emotional_tone: uno tra {tones}\n"
    "- brand_entities: elenco di marchi/aziende citati esplicitamente "
    "(vuoto se nessuno)."
)

# --------------------------------------------------------------------------- #
# Phase 3 — Community Vibe Aggregator (TEXT-ONLY, Gemini 2.5 Pro)
#   placeholders: {platform} {topic} {tone} {comments}
# --------------------------------------------------------------------------- #
COMMUNITY_VIBE_PROMPT = (
    "Sei un sociologo delle community online. Di seguito un campione di "
    "commenti (di primo livello e risposte) sotto un singolo post su "
    "{platform}. Contesto del post: argomento='{topic}', tono='{tone}'.\n\n"
    "--- COMMENTI CAMPIONE ---\n{comments}\n\n"
    "Valuta il 'vibe' collettivo e restituisci ESCLUSIVAMENTE JSON conforme "
    "allo schema (nessun markdown):\n"
    "- sentiment_polarization_index: numero 0.0-1.0 (0.0=consenso totale, "
    "1.0=frammentazione/polarizzazione marcata)\n"
    "- dominant_community_emotion: emozione modale del pubblico\n"
    "- community_noun_phrases: frasi nominali salienti su cui converge il "
    "pubblico."
)

# --------------------------------------------------------------------------- #
# Phase 1 (Instagram) — MULTIMODAL Post Summarizer (frames + transcript + caption)
#   placeholders: {subtype} {caption} {transcript} {formats} {tones}
# --------------------------------------------------------------------------- #
MULTIMODAL_POST_PROMPT = (
    "Analizza questo post Instagram di tipo '{subtype}'. Ti fornisco i FRAME "
    "estratti dal contenuto, la trascrizione audio e la didascalia.\n\n"
    "--- DIDASCALIA ---\n{caption}\n\n"
    "--- TRASCRIZIONE ---\n{transcript}\n\n"
    "Osserva i frame e combinali con il testo. Restituisci SOLO JSON conforme "
    "allo schema (nessun markdown):\n"
    "- format_type: uno tra {formats}\n"
    "- primary_topic: argomento principale (max 6 parole)\n"
    "- intended_emotional_tone: uno tra {tones}\n"
    "- brand_entities: marchi/aziende visibili o citati (vuoto se nessuno)\n"
    "- visual_summary: cosa si vede nei frame (1-2 frasi)\n"
    "- on_screen_text: testo a schermo / sticker (vuoto se nessuno)\n"
    "- visual_setting: ambientazione (es. cucina, esterno città, studio)."
)

# --------------------------------------------------------------------------- #
# Phase 3 (Instagram) — Cache-Augmented Generation
#   CAG_SYSTEM_INSTRUCTION  : system instruction stored in the CachedContent
#                             (no placeholders)
#   CAG_COMMENT_CHUNK_PROMPT : per-chunk query against the cache
#                             placeholders: {comments}
# --------------------------------------------------------------------------- #
CAG_SYSTEM_INSTRUCTION = (
    "Sei un sociologo delle community. Le immagini (frame del post), "
    "la trascrizione e la didascalia forniti sono la VERITÀ DI BASE "
    "del post Instagram. Userai SOLO questo contesto per valutare i "
    "commenti che ti verranno passati."
)

CAG_COMMENT_CHUNK_PROMPT = (
    "Sulla base del contesto del post fornito (frame, trascrizione, "
    "didascalia in cache), valuta SOLO questi commenti e restituisci JSON:\n"
    "{comments}\n\n"
    "- sentiment_polarization_index: 0.0 (consenso) → 1.0 (frammentazione)\n"
    "- dominant_community_emotion: emozione modale\n"
    "- community_noun_phrases: frasi nominali salienti dei commenti\n"
    "- visual_reference_ratio: frazione (0-1) di commenti che fanno "
    "riferimento a ciò che si VEDE nei frame del post."
)


# --------------------------------------------------------------------------- #
# Discovery registry — every prompt by name.
# --------------------------------------------------------------------------- #
PROMPTS: Dict[str, str] = {
    "post_context": POST_CONTEXT_PROMPT,
    "community_vibe": COMMUNITY_VIBE_PROMPT,
    "multimodal_post": MULTIMODAL_POST_PROMPT,
    "cag_system_instruction": CAG_SYSTEM_INSTRUCTION,
    "cag_comment_chunk": CAG_COMMENT_CHUNK_PROMPT,
}


if __name__ == "__main__":  # quick listing / sanity check
    for _name, _text in PROMPTS.items():
        print(f"\n=== {_name} ===\n{_text}")


In [ ]:
"""Show Reel Media Group — Post-Level Context Enrichment & Community Vibe Baseline."""

from __future__ import annotations

import json
import logging
import re
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd

from prompts import COMMUNITY_VIBE_PROMPT, MULTIMODAL_POST_PROMPT, POST_CONTEXT_PROMPT

try:
    import vertexai
    from vertexai.generative_models import GenerationConfig, GenerativeModel, Part
    _HAS_VERTEX = True
except Exception:
    _HAS_VERTEX = False

try:
    import spacy
    _HAS_SPACY = True
except Exception:
    _HAS_SPACY = False

try:
    import emoji as emoji_lib
    _HAS_EMOJI = True
except Exception:
    _HAS_EMOJI = False

try:
    from pydantic import BaseModel, Field, ValidationError, field_validator
    _HAS_PYDANTIC = True
except Exception:
    _HAS_PYDANTIC = False


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
LOGGER = logging.getLogger("community_vibe_pipeline")


@dataclass(frozen=True)
class PipelineConfig:
    gcp_project_id: str = "gen-lang-client-0792749758"
    gcp_location: str = "us-central1"
    flash_model: str = "gemini-2.5-flash"
    pro_model: str = "gemini-2.5-pro"
    spacy_model: str = "it_core_news_lg"
    post_batch_size: int = 32
    max_comments_sampled: int = 60
    min_comments_for_vibe: int = 3
    llm_max_retries: int = 3
    llm_temperature: float = 0.1
    output_path: str = "gs://afb_showreel/enriched/enriched_post_vibe_matrix.parquet"
    enable_llm: bool = True


CANON_MEDIA_ID    = "media_id"
CANON_POST_TEXT   = "post_text"
CANON_TRANSCRIPT  = "transcript"
CANON_PLATFORM    = "platform"
CANON_COMMENT_ID  = "comment_id"
CANON_PARENT_ID   = "parent_id"
CANON_AUTHOR_ID   = "author_id"
CANON_COMMENT_TEXT = "comment_text"
CANON_TIMESTAMP   = "timestamp"
CANON_MUSIC_INFO  = "music_info"

# Columns attached by the multimodal join; stripped after Phase 1.
# music_info is NOT in this list — it's a real output column.
_INTERNAL_COLS = ("_frame_uris", "_image_uris", "_mm_subtype")

MEDIA_COLUMN_MAP: Dict[str, Dict[str, Sequence[str]]] = {
    "instagram": {
        CANON_MEDIA_ID:   ("media_id",),
        CANON_POST_TEXT:  ("caption", "message"),
        CANON_TRANSCRIPT: ("transcript",),
        CANON_TIMESTAMP:  ("timestamp",),
        CANON_MUSIC_INFO: ("music_info",),   # joined from .info.json via multimodal assets
    },
    "facebook": {
        CANON_MEDIA_ID:   ("post_id", "media_id"),
        CANON_POST_TEXT:  ("message", "caption"),
        CANON_TRANSCRIPT: ("transcript",),
        CANON_TIMESTAMP:  ("timestamp",),
    },
    "tiktok": {
        CANON_MEDIA_ID:   ("media_id", "video_id", "post_id"),
        CANON_POST_TEXT:  ("caption", "description", "message"),
        CANON_TRANSCRIPT: ("transcript",),
        CANON_TIMESTAMP:  ("timestamp",),
    },
    "youtube": {
        CANON_MEDIA_ID:   ("video_id", "media_id"),
        CANON_POST_TEXT:  ("title", "description"),
        CANON_TRANSCRIPT: ("transcript", "captions"),
        CANON_TIMESTAMP:  ("timestamp", "published_at"),
    },
}

COMMENT_COLUMN_MAP: Dict[str, Dict[str, Sequence[str]]] = {
    "instagram": {
        CANON_COMMENT_ID:   ("comment_id",),
        CANON_MEDIA_ID:     ("media_id",),
        CANON_PARENT_ID:    ("parent_id", "reply_to_comment_id"),
        CANON_AUTHOR_ID:    ("from_id", "author_id"),
        CANON_COMMENT_TEXT: ("text", "message"),
        CANON_TIMESTAMP:    ("timestamp",),
    },
    "facebook": {
        CANON_COMMENT_ID:   ("comment_id",),
        CANON_MEDIA_ID:     ("post_id", "media_id"),
        CANON_PARENT_ID:    ("parent_id", "reply_to_comment_id"),
        CANON_AUTHOR_ID:    ("from_id", "author_id"),
        CANON_COMMENT_TEXT: ("message", "text"),
        CANON_TIMESTAMP:    ("timestamp",),
    },
    "tiktok": {
        CANON_COMMENT_ID:   ("comment_id",),
        CANON_MEDIA_ID:     ("media_id", "video_id"),
        CANON_PARENT_ID:    ("reply_id", "parent_id"),
        CANON_AUTHOR_ID:    ("uid", "from_id", "author_id"),
        CANON_COMMENT_TEXT: ("text", "message"),
        CANON_TIMESTAMP:    ("timestamp",),
    },
}

ALLOWED_FORMATS = (
    "Sponsored Skit", "Organic Vlog", "Product Review", "Tutorial", "Q&A",
    "Behind The Scenes", "News Commentary", "Meme", "Announcement", "Other",
)
ALLOWED_TONES = (
    "Humorous", "Inspirational", "Informative", "Nostalgic",
    "Provocative", "Heartfelt", "Neutral", "Promotional",
)

if _HAS_PYDANTIC:
    class PostContext(BaseModel):
        format_type: str = Field(...)
        primary_topic: str = Field(...)
        intended_emotional_tone: str = Field(...)
        brand_entities: List[str] = Field(default_factory=list)

        @field_validator("brand_entities", mode="before")
        @classmethod
        def _coerce_list(cls, v: Any) -> List[str]:
            if v is None: return []
            if isinstance(v, str): return [v] if v.strip() else []
            return list(v)

    class CommunityVibe(BaseModel):
        sentiment_polarization_index: float = Field(..., ge=0.0, le=1.0)
        dominant_community_emotion: str
        community_noun_phrases: List[str] = Field(default_factory=list)

        @field_validator("sentiment_polarization_index", mode="before")
        @classmethod
        def _clamp(cls, v: Any) -> float:
            try: return float(min(1.0, max(0.0, float(v))))
            except: return 0.5

        @field_validator("community_noun_phrases", mode="before")
        @classmethod
        def _coerce_list(cls, v: Any) -> List[str]:
            if v is None: return []
            if isinstance(v, str): return [v] if v.strip() else []
            return list(v)
else:
    PostContext = dict  # type: ignore
    CommunityVibe = dict  # type: ignore

POST_CONTEXT_RESPONSE_SCHEMA: Dict[str, Any] = {
    "type": "OBJECT",
    "properties": {
        "format_type":             {"type": "STRING", "enum": list(ALLOWED_FORMATS)},
        "primary_topic":           {"type": "STRING"},
        "intended_emotional_tone": {"type": "STRING", "enum": list(ALLOWED_TONES)},
        "brand_entities":          {"type": "ARRAY", "items": {"type": "STRING"}},
    },
    "required": ["format_type", "primary_topic", "intended_emotional_tone", "brand_entities"],
}
COMMUNITY_VIBE_RESPONSE_SCHEMA: Dict[str, Any] = {
    "type": "OBJECT",
    "properties": {
        "sentiment_polarization_index": {"type": "NUMBER"},
        "dominant_community_emotion":   {"type": "STRING"},
        "community_noun_phrases":       {"type": "ARRAY", "items": {"type": "STRING"}},
    },
    "required": ["sentiment_polarization_index", "dominant_community_emotion", "community_noun_phrases"],
}


# --------------------------------------------------------------------------- #
# Schema normalization
# --------------------------------------------------------------------------- #
class SchemaNormalizer:
    @staticmethod
    def _resolve(df: pd.DataFrame, candidates: Sequence[str]) -> Optional[str]:
        for name in candidates:
            if name in df.columns:
                return name
        return None

    @classmethod
    def _normalize(cls, df, platform, column_map, required):
        if platform not in column_map:
            raise KeyError(f"Unsupported platform '{platform}'")
        mapping = column_map[platform]
        out = pd.DataFrame(index=df.index)
        for canon, candidates in mapping.items():
            source = cls._resolve(df, candidates)
            out[canon] = df[source] if source is not None else pd.NA
        out[CANON_PLATFORM] = platform
        for col in _INTERNAL_COLS:
            if col in df.columns:
                out[col] = df[col]
        missing = [c for c in required if out[c].isna().all()]
        if missing:
            LOGGER.warning("[%s] required canonical columns absent: %s", platform, missing)
        for col in out.columns:
            if col not in _INTERNAL_COLS and (
                out[col].dtype == object or col.endswith("_id") or "text" in col
            ):
                out[col] = out[col].astype("string")
        return out

    @classmethod
    def normalize_media(cls, df, platform):
        norm = cls._normalize(df, platform, MEDIA_COLUMN_MAP, [CANON_MEDIA_ID, CANON_POST_TEXT])
        before = len(norm)
        norm = norm.dropna(subset=[CANON_MEDIA_ID]).reset_index(drop=True)
        LOGGER.info("[%s] media: %d → %d rows", platform, before, len(norm))
        return norm

    @classmethod
    def normalize_comments(cls, df, platform):
        norm = cls._normalize(df, platform, COMMENT_COLUMN_MAP, [CANON_MEDIA_ID, CANON_COMMENT_TEXT])
        before = len(norm)
        norm = norm.dropna(subset=[CANON_MEDIA_ID, CANON_COMMENT_TEXT]).reset_index(drop=True)
        LOGGER.info("[%s] comments: %d → %d rows", platform, before, len(norm))
        return norm


# --------------------------------------------------------------------------- #
# Vertex AI client
# --------------------------------------------------------------------------- #
class VertexLLMClient:
    def __init__(self, config: PipelineConfig) -> None:
        self.config = config
        self._flash = self._pro = None
        self.available = False
        self.call_count = self.error_count = 0
        if not (config.enable_llm and _HAS_VERTEX):
            LOGGER.warning("Vertex AI disabled/unavailable — using deterministic fallbacks.")
            return
        try:
            vertexai.init(project=config.gcp_project_id, location=config.gcp_location)
            self._flash = GenerativeModel(config.flash_model)
            self._pro   = GenerativeModel(config.pro_model)
            self.available = True
            LOGGER.info("Vertex AI initialized (project=%s).", config.gcp_project_id)
        except Exception as exc:
            LOGGER.error("Vertex AI init failed: %s", exc)

    def _generate(self, model, content: Union[str, list], schema) -> Optional[Dict]:
        cfg = GenerationConfig(
            temperature=self.config.llm_temperature,
            response_mime_type="application/json",
            response_schema=schema,
        )
        for attempt in range(1, self.config.llm_max_retries + 1):
            try:
                self.call_count += 1
                return json.loads(model.generate_content(content, generation_config=cfg).text)
            except json.JSONDecodeError as e:
                LOGGER.warning("JSON decode (attempt %d): %s", attempt, e)
            except Exception as e:
                LOGGER.warning("Gemini call (attempt %d): %s", attempt, e)
        self.error_count += 1
        return None

    def summarize_post(self, content: Union[str, list]) -> Optional[Dict]:
        if not self.available or self._flash is None: return None
        return self._generate(self._flash, content, POST_CONTEXT_RESPONSE_SCHEMA)

    def assess_community(self, prompt: str) -> Optional[Dict]:
        if not self.available or self._pro is None: return None
        return self._generate(self._pro, prompt, COMMUNITY_VIBE_RESPONSE_SCHEMA)


# --------------------------------------------------------------------------- #
# Phase 1 — Post-Level Context Summarizer
# --------------------------------------------------------------------------- #
_MIME_MAP = {".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".png": "image/png", ".webp": "image/webp"}


class PostContextSummarizer:
    """Phase 1: text-only prompt for all platforms; multimodal for IG posts with assets."""

    def __init__(self, config: PipelineConfig, llm: VertexLLMClient) -> None:
        self.config = config
        self.llm = llm

    def _build_prompt(self, row: pd.Series) -> str:
        t = row.get(CANON_TRANSCRIPT); t = "" if pd.isna(t) else str(t)[:8000]
        p = row.get(CANON_POST_TEXT);  p = "" if pd.isna(p) else str(p)[:4000]
        return POST_CONTEXT_PROMPT.format(
            platform=row.get(CANON_PLATFORM, "social"),
            post_text=p or "(nessuna didascalia)",
            transcript=t or "(nessuna trascrizione)",
            formats=", ".join(ALLOWED_FORMATS),
            tones=", ".join(ALLOWED_TONES),
        )

    def _build_content(self, row: pd.Series) -> Union[str, list]:
        frame_uris = row.get("_frame_uris") or []
        image_uris = row.get("_image_uris") or []
        all_uris   = list(frame_uris) + list(image_uris)
        if not (all_uris and _HAS_VERTEX and self.llm.available):
            return self._build_prompt(row)

        caption  = row.get(CANON_POST_TEXT);  caption  = "" if pd.isna(caption)  else str(caption)[:4000]
        transcript = row.get(CANON_TRANSCRIPT); transcript = "" if pd.isna(transcript) else str(transcript)[:8000]
        subtype  = row.get("_mm_subtype") or "feed"
        if pd.isna(subtype): subtype = "feed"

        text_part = MULTIMODAL_POST_PROMPT.format(
            subtype=str(subtype),
            caption=caption or "(nessuna didascalia)",
            transcript=transcript or "(nessuna trascrizione)",
            formats=", ".join(ALLOWED_FORMATS),
            tones=", ".join(ALLOWED_TONES),
        )
        content: list = []
        for uri in all_uris:
            ext  = "." + uri.rsplit(".", 1)[-1].lower() if "." in uri else ""
            mime = _MIME_MAP.get(ext, "image/jpeg")
            try:
                content.append(Part.from_uri(uri, mime_type=mime))
            except Exception as e:
                LOGGER.debug("Part.from_uri failed for %s: %s", uri, e)
        content.append(text_part)
        return content if len(content) > 1 else self._build_prompt(row)

    @staticmethod
    def _fallback(row: pd.Series) -> Dict:
        text = " ".join(str(row.get(c, "") or "") for c in (CANON_POST_TEXT, CANON_TRANSCRIPT)).lower()
        sponsored = any(k in text for k in ("#ad", "sponsor", "adv", "in collaborazione"))
        return {"format_type": "Sponsored Skit" if sponsored else "Organic Vlog",
                "primary_topic": "unknown", "intended_emotional_tone": "Neutral", "brand_entities": []}

    def _validate(self, payload, row) -> Dict:
        if payload is None: return self._fallback(row)
        if _HAS_PYDANTIC:
            try: return PostContext(**payload).model_dump()
            except ValidationError as e:
                LOGGER.warning("PostContext validation failed: %s", e)
                return self._fallback(row)
        return {**self._fallback(row), **payload}

    def run(self, media_df: pd.DataFrame) -> pd.DataFrame:
        n_mm = (
            (media_df.get("_frame_uris", pd.Series(dtype=object)).apply(bool)
             | media_df.get("_image_uris", pd.Series(dtype=object)).apply(bool)).sum()
            if "_frame_uris" in media_df.columns else 0
        )
        LOGGER.info("Phase 1: %d posts (batch=%d) — %d multimodal.",
                    len(media_df), self.config.post_batch_size, n_mm)
        records = []
        for start in range(0, len(media_df), self.config.post_batch_size):
            for _, row in media_df.iloc[start:start + self.config.post_batch_size].iterrows():
                records.append(self._validate(self.llm.summarize_post(self._build_content(row)), row))
            LOGGER.info("Phase 1: %d/%d done.", min(start + self.config.post_batch_size, len(media_df)), len(media_df))
        ctx = pd.DataFrame.from_records(records, index=media_df.index)
        result = pd.concat([media_df, ctx], axis=1)
        return result.drop(columns=list(_INTERNAL_COLS), errors="ignore")


# --------------------------------------------------------------------------- #
# Phase 2 — High-Velocity Local NLP Enrichment
# --------------------------------------------------------------------------- #
_EMOJI_REGEX = re.compile(
    "[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F000-\U0001F0FF"
    "\U00002700-\U000027BF\U0001F900-\U0001F9FF]+"
)
_PUNCT_REGEX = re.compile(r"[!?.,;:…]")
_WORD_REGEX  = re.compile(r"\b\w+\b", re.UNICODE)


class LocalNLPEnricher:
    """Phase 2: spaCy vector similarity + token metrics.

    Post vector is built from four text signals (richest → most disambiguating):
      1. post_text / caption          — the authored message
      2. primary_topic                — Phase-1 LLM distillation
      3. transcript (first 500 chars) — spoken content of the video
      4. music_info                   — song + artist; music mood primes comment tone

    Transcript is capped at 500 chars because spaCy averages all token vectors;
    a full transcript would dilute the caption's signal. music_info is short
    by nature so no cap is needed.
    """

    _TRANSCRIPT_CHARS = 500   # how much of the transcript to include in the post vector

    def __init__(self, config: PipelineConfig) -> None:
        self.config = config
        self.nlp = self._load_model(config.spacy_model)
        self.has_vectors = bool(self.nlp) and self.nlp.vocab.vectors_length > 0  # type: ignore
        if self.nlp is None:
            LOGGER.warning("spaCy model '%s' unavailable; using lexical fallbacks.", config.spacy_model)

    @staticmethod
    def _load_model(name):
        if not _HAS_SPACY: return None
        try: return spacy.load(name)
        except Exception as e:
            LOGGER.error("spaCy load failed: %s", e)
            return None

    @staticmethod
    def _extract_emojis(text):
        if _HAS_EMOJI: return [e["emoji"] for e in emoji_lib.emoji_list(text)]
        return _EMOJI_REGEX.findall(text)

    @classmethod
    def token_metrics(cls, text):
        text = text or ""
        words = _WORD_REGEX.findall(text)
        wc = len(words); ec = len(cls._extract_emojis(text))
        pc = len(_PUNCT_REGEX.findall(text)); cc = max(len(text), 1)
        return {
            "word_count": float(wc), "char_count": float(len(text)),
            "emoji_count": float(ec), "emoji_density": float(ec / (wc + 1)),
            "punctuation_velocity": float(pc / cc),
            "avg_word_length": float(np.mean([len(w) for w in words])) if words else 0.0,
        }

    def _doc(self, text):
        if self.nlp is None or not text: return None
        try: return self.nlp(text[:10000])
        except Exception as e:
            LOGGER.debug("spaCy parse: %s", e); return None

    def vector(self, text):
        doc = self._doc(text)
        if doc is not None and self.has_vectors and doc.has_vector:
            return np.asarray(doc.vector, dtype=np.float32)
        return np.zeros(self.nlp.vocab.vectors_length if self.has_vectors else 1, dtype=np.float32)  # type: ignore

    def noun_set(self, text):
        doc = self._doc(text)
        if doc is None:
            return {w.lower() for w in _WORD_REGEX.findall(text or "") if len(w) > 3}
        nouns = {t.lemma_.lower() for t in doc if t.pos_ in ("NOUN", "PROPN") and not t.is_stop}
        nouns |= {c.root.lemma_.lower() for c in doc.noun_chunks}
        return {n for n in nouns if n.strip()}

    def entity_set(self, text):
        doc = self._doc(text)
        if doc is None: return set()
        return {e.text.lower().strip() for e in doc.ents if e.text.strip()}

    @staticmethod
    def cosine_similarity(u, v):
        nu, nv = np.linalg.norm(u), np.linalg.norm(v)
        if nu == 0.0 or nv == 0.0: return 0.0
        return float(np.clip(np.dot(u, v) / (nu * nv), -1.0, 1.0))

    @staticmethod
    def jaccard(a, b):
        if not a and not b: return 0.0
        union = a | b
        return float(len(a & b) / len(union)) if union else 0.0

    def enrich_comments(self, comments_df: pd.DataFrame, post_context: pd.DataFrame) -> pd.DataFrame:
        """Enrich each comment with NLP features keyed against its parent post.

        Post vector is built from: caption + primary_topic + transcript[:500] + music_info.
        Transcript and music are absent for non-IG platforms, so they default to "".
        """
        LOGGER.info("Phase 2: enriching %d comments.", len(comments_df))

        # Build post-side context text: caption + topic + transcript + music
        post_text_col = post_context[CANON_POST_TEXT].fillna("")
        topic_col     = post_context.get("primary_topic",  pd.Series("", index=post_context.index)).fillna("")
        transcript_col = (
            post_context[CANON_TRANSCRIPT].fillna("").apply(
                lambda t: str(t)[:self._TRANSCRIPT_CHARS]
            ) if CANON_TRANSCRIPT in post_context.columns
            else pd.Series("", index=post_context.index)
        )
        music_col = (
            post_context[CANON_MUSIC_INFO].fillna("")
            if CANON_MUSIC_INFO in post_context.columns
            else pd.Series("", index=post_context.index)
        )

        # Four signals concatenated — richest semantic representation of the post.
        post_ctx_text = (
            post_text_col + " "
            + topic_col + " "
            + transcript_col + " "
            + music_col
        )

        brand_col = post_context.get("brand_entities", pd.Series([[]] * len(post_context), index=post_context.index))

        post_vectors: Dict[str, np.ndarray] = {}
        post_entities: Dict[str, set] = {}
        post_nouns: Dict[str, set] = {}
        for media_id, ctx_text, body_text, brands in zip(
            post_context[CANON_MEDIA_ID], post_ctx_text, post_text_col, brand_col
        ):
            mid = str(media_id)
            post_vectors[mid] = self.vector(ctx_text)
            ents = self.entity_set(str(body_text))
            if isinstance(brands, (list, tuple)):
                ents |= {str(b).lower() for b in brands}
            post_entities[mid] = ents
            post_nouns[mid] = self.noun_set(ctx_text)

        out_rows = []
        for _, row in comments_df.iterrows():
            text = "" if pd.isna(row[CANON_COMMENT_TEXT]) else str(row[CANON_COMMENT_TEXT])
            mid  = str(row[CANON_MEDIA_ID])
            metrics = self.token_metrics(text)
            c_vec  = self.vector(text); c_nouns = self.noun_set(text)
            p_vec  = post_vectors.get(mid, np.zeros_like(c_vec))
            p_ents = post_entities.get(mid, set()); p_nouns = post_nouns.get(mid, set())
            overlap = len(c_nouns & p_ents)
            metrics.update({
                CANON_COMMENT_ID: row.get(CANON_COMMENT_ID),
                CANON_MEDIA_ID: mid, CANON_PLATFORM: row.get(CANON_PLATFORM),
                "entity_overlap_count": float(overlap),
                "entity_overlap_ratio": float(overlap / (len(c_nouns) or 1)),
                "context_cosine_similarity": self.cosine_similarity(c_vec, p_vec),
                "noun_jaccard_vs_post": self.jaccard(c_nouns, p_nouns),
                "_noun_set": sorted(c_nouns),
            })
            out_rows.append(metrics)

        enriched = pd.DataFrame.from_records(out_rows)
        fc = enriched.select_dtypes(include="float").columns
        enriched[fc] = enriched[fc].apply(pd.to_numeric, downcast="float")
        LOGGER.info("Phase 2: produced %d enriched comment rows.", len(enriched))
        return enriched


# --------------------------------------------------------------------------- #
# Phase 3 — Community Vibe & Polarization Aggregator
# --------------------------------------------------------------------------- #
class CommunityVibeAggregator:
    def __init__(self, config, llm, nlp) -> None:
        self.config = config; self.llm = llm; self.nlp = nlp

    @staticmethod
    def _sample_comments(group, k):
        if len(group) <= k: return group
        is_reply = group[CANON_PARENT_ID].notna() if CANON_PARENT_ID in group else pd.Series(False, index=group.index)
        top = group[~is_reply]; replies = group[is_reply]
        n_top = min(len(top), max(k // 2, k - len(replies))); n_rep = k - n_top
        return pd.concat([
            top.sample(n=n_top, random_state=42) if n_top else top.head(0),
            replies.sample(n=min(n_rep, len(replies)), random_state=42) if n_rep else replies.head(0),
        ])

    def _build_prompt(self, sample, context):
        lines = "\n".join(f"- {str(t)[:300]}" for t in sample[CANON_COMMENT_TEXT].dropna())
        return COMMUNITY_VIBE_PROMPT.format(
            platform=context.get(CANON_PLATFORM, "social"),
            topic=context.get("primary_topic", "n/a"),
            tone=context.get("intended_emotional_tone", "n/a"),
            comments=lines or "(nessun commento)",
        )

    @staticmethod
    def _polarization_fallback(metrics):
        sims = metrics["context_cosine_similarity"].to_numpy(dtype=float)
        if sims.size < 2: return 0.5
        return float(np.clip(np.std(sims) / 0.5, 0.0, 1.0))

    def run(self, enriched_comments, post_context, raw_comments):
        LOGGER.info("Phase 3: %d media.", post_context[CANON_MEDIA_ID].nunique())
        agg = enriched_comments.groupby(CANON_MEDIA_ID).agg(
            comment_count=(CANON_COMMENT_ID, "count"),
            mean_cosine_similarity=("context_cosine_similarity", "mean"),
            mean_entity_overlap=("entity_overlap_count", "mean"),
            mean_emoji_density=("emoji_density", "mean"),
            mean_punctuation_velocity=("punctuation_velocity", "mean"),
            mean_noun_jaccard=("noun_jaccard_vs_post", "mean"),
        )
        comment_nouns: Dict[str, set] = {}
        if "_noun_set" in enriched_comments.columns:
            for mid, sub in enriched_comments.groupby(CANON_MEDIA_ID)["_noun_set"]:
                bag: set = set()
                for s in sub: bag |= set(s)
                comment_nouns[str(mid)] = bag

        ctx_idx = post_context.set_index(CANON_MEDIA_ID)
        raw_by  = dict(tuple(raw_comments.groupby(CANON_MEDIA_ID)))
        records = []
        for media_id, struct in agg.iterrows():
            mid = str(media_id)
            ctx = ctx_idx.loc[media_id] if media_id in ctx_idx.index else pd.Series(dtype=object)
            ms  = enriched_comments[enriched_comments[CANON_MEDIA_ID] == mid]
            p_nouns = self.nlp.noun_set(str(ctx.get(CANON_POST_TEXT, "") or ""))
            adherence  = self.nlp.jaccard(comment_nouns.get(mid, set()), p_nouns)
            polariz    = self._polarization_fallback(ms)
            emotion    = "mixed"
            phrases    = sorted(comment_nouns.get(mid, set()))[:15]
            grp = raw_by.get(media_id)
            if grp is not None and len(grp) >= self.config.min_comments_for_vibe:
                payload = self.llm.assess_community(
                    self._build_prompt(self._sample_comments(grp, self.config.max_comments_sampled), ctx)
                )
                if payload is not None:
                    v = self._validate(payload)
                    polariz = v["sentiment_polarization_index"]
                    emotion = v["dominant_community_emotion"]
                    if v["community_noun_phrases"]:
                        adherence = max(adherence, self.nlp.jaccard(
                            {p.lower() for p in v["community_noun_phrases"]}, p_nouns))
                        phrases = v["community_noun_phrases"]
            rec = {CANON_MEDIA_ID: mid, CANON_PLATFORM: ctx.get(CANON_PLATFORM, pd.NA),
                   "sentiment_polarization_index": polariz, "topical_adherence_score": adherence,
                   "dominant_community_emotion": emotion, "community_noun_phrases": phrases}
            rec.update(struct.to_dict()); records.append(rec)
        LOGGER.info("Phase 3: done.")
        return pd.DataFrame.from_records(records)

    def _validate(self, payload):
        if _HAS_PYDANTIC:
            try: return CommunityVibe(**payload).model_dump()
            except ValidationError as e: LOGGER.warning("CommunityVibe validation: %s", e)
        return {
            "sentiment_polarization_index": float(payload.get("sentiment_polarization_index", 0.5)),
            "dominant_community_emotion":   str(payload.get("dominant_community_emotion", "mixed")),
            "community_noun_phrases":       list(payload.get("community_noun_phrases", []) or []),
        }


# --------------------------------------------------------------------------- #
# Orchestrator
# --------------------------------------------------------------------------- #
class EnrichmentPipeline:
    def __init__(self, config=None) -> None:
        self.config     = config or PipelineConfig()
        self.llm        = VertexLLMClient(self.config)
        self.nlp        = LocalNLPEnricher(self.config)
        self.summarizer = PostContextSummarizer(self.config, self.llm)
        self.aggregator = CommunityVibeAggregator(self.config, self.llm, self.nlp)

    @staticmethod
    def _concat_normalized(frames, normalizer):
        parts = [normalizer(df, p) for p, df in frames.items() if df is not None and len(df)]
        return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

    def run(self, media_frames, comment_frames):
        LOGGER.info("=== Enrichment pipeline START ===")
        media    = self._concat_normalized(media_frames,   SchemaNormalizer.normalize_media)
        comments = self._concat_normalized(comment_frames, SchemaNormalizer.normalize_comments)
        if media.empty: raise ValueError("No media rows after normalization.")
        LOGGER.info("Normalized: %d media, %d comments.", len(media), len(comments))

        post_context = self.summarizer.run(media)

        if comments.empty:
            LOGGER.warning("No comments; Phase 2/3 skipped.")
            enriched_comments = pd.DataFrame(columns=[
                CANON_MEDIA_ID, CANON_COMMENT_ID, "context_cosine_similarity",
                "entity_overlap_count", "emoji_density", "punctuation_velocity", "noun_jaccard_vs_post",
            ])
        else:
            enriched_comments = self.nlp.enrich_comments(comments, post_context)

        vibe = (
            self.aggregator.run(enriched_comments, post_context, comments)
            if not enriched_comments.empty else pd.DataFrame(columns=[CANON_MEDIA_ID])
        )
        matrix = post_context.merge(vibe, on=[CANON_MEDIA_ID], how="left", suffixes=("", "_vibe"))
        matrix = matrix.drop(columns=[CANON_TRANSCRIPT], errors="ignore")
        self._persist(matrix)
        LOGGER.info("=== Pipeline DONE === rows=%d | LLM calls=%d | errors=%d",
                    len(matrix), self.llm.call_count, self.llm.error_count)
        return matrix

    def _persist(self, matrix):
        out = str(self.config.output_path)
        df  = matrix.copy()
        for col in df.columns:
            if df[col].apply(lambda x: isinstance(x, (list, tuple, set))).any():
                df[col] = df[col].apply(
                    lambda x: json.dumps(list(x), ensure_ascii=False) if isinstance(x, (list, tuple, set)) else x)
        try:
            df.to_parquet(out, engine="pyarrow", compression="snappy", index=False)
            LOGGER.info("Wrote → %s", out)
        except Exception as exc:
            csv_out = (out[:-8] + ".csv") if out.endswith(".parquet") else out + ".csv"
            LOGGER.error("Parquet failed (%s); CSV → %s", exc, csv_out)
            df.to_csv(csv_out, index=False)


# --------------------------------------------------------------------------- #
# Mock data for smoke tests
# --------------------------------------------------------------------------- #
def _mock_frames() -> Tuple[Dict[str, pd.DataFrame], Dict[str, pd.DataFrame]]:
    ig_media = pd.DataFrame({
        "media_id":   ["IG1", "IG2"],
        "caption":    ["Nuovo video con @nike! Routine mattutina 🏃‍♀️ #ad",
                        "Riflessioni della domenica ☀️"],
        "transcript": ["Ciao ragazzi oggi proviamo le nuove scarpe Nike.", ""],
        "timestamp":  ["2026-03-01T10:00:00Z", "2026-03-02T18:30:00Z"],
        "music_info": ["Running Up That Hill — Kate Bush", ""],
        "_frame_uris": [[], []], "_image_uris": [[], []], "_mm_subtype": ["feed", "feed"],
    })
    fb_posts = pd.DataFrame({
        "post_id": ["FB1"], "message": ["Apriamo a Milano! 🎉"], "timestamp": ["2026-03-03T09:00:00Z"],
    })
    tk_media = pd.DataFrame({
        "video_id": ["TK1"], "caption": ["POV: caffè finito 😂 #comedy"], "timestamp": ["2026-03-04T12:00:00Z"],
    })
    ig_comments = pd.DataFrame({
        "comment_id": ["c1","c2","c3","c4"], "media_id": ["IG1","IG1","IG1","IG2"],
        "from_id": ["u1","u2","u3","u4"],
        "text": ["Adoro le Nike! ❤️", "Super motivante 💪", "Troppa pub 🙄", "Che dolce ☀️"],
        "parent_id": [None,None,"c1",None], "timestamp": ["2026-03-01T11:00:00Z"]*4,
    })
    fb_comments = pd.DataFrame({
        "comment_id": ["f1","f2","f3"], "post_id": ["FB1"]*3, "from_id": ["u5","u6","u7"],
        "message": ["Finalmente a Milano! 🎉","Speriamo Roma","Non mi interessa"],
        "parent_id": [None,None,None], "timestamp": ["2026-03-03T10:00:00Z"]*3,
    })
    tk_comments = pd.DataFrame({
        "comment_id": ["t1","t2","t3"], "video_id": ["TK1"]*3, "uid": ["u8","u9","u10"],
        "text": ["HAHAHA 😂","Troppo vero 💀","Il caffè è vita ☕"],
        "reply_id": [None,"t1",None], "timestamp": ["2026-03-04T13:00:00Z"]*3,
    })
    return (
        {"instagram": ig_media, "facebook": fb_posts, "tiktok": tk_media},
        {"instagram": ig_comments, "facebook": fb_comments, "tiktok": tk_comments},
    )

In [ ]:
# Cell 6 — GCS data loader (pandas reads gs:// natively via gcsfs)
import json as _json
import re as _re
import gcsfs as _gcsfs

MEDIA_FILES = {
    "instagram": "ig_posts_cleaned.parquet",
    "facebook":  "fb_posts_clean.parquet",
    "tiktok":    "tk_posts_clean.parquet",
    "youtube":   "videos_metadata.csv",
}
COMMENT_FILES = {
    "instagram": "ig_comments_cleaned.parquet",
    "facebook":  "fb_comments_clean.parquet",
    "tiktok":    "tk_comments_clean.parquet",
}
YT_COMMENT_FILES = [
    "YTcomments_1_cleaned.parquet",
    "YTcomments_2_cleaned.parquet",
    "YTcomments_3_cleaned.parquet",
    "YTcomments_4_cleaned.parquet",
]

_IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}


def _read(uri: str) -> "pd.DataFrame":
    if uri.endswith(".parquet"):
        return pd.read_parquet(uri)
    return pd.read_csv(uri)


def _extract_shortcode(permalink) -> str:
    if not isinstance(permalink, str):
        return ""
    m = _re.search(r'/(?:p|reel|tv)/([A-Za-z0-9_-]+)', permalink)
    return m.group(1) if m else ""


def _is_image(path: str) -> bool:
    return any(path.lower().endswith(ext) for ext in _IMG_EXTS)


def _parse_music(info: dict) -> str:
    """Extract a human-readable music string from a yt-dlp .info.json dict.

    yt-dlp stores Instagram music metadata at the top level (music_song /
    music_artist) or sometimes under an 'extras' sub-dict. Some versions also
    use the generic 'track' / 'artist' keys used for music videos.
    Returns 'Song — Artist' when both are present, or whichever is available.
    """
    # Primary yt-dlp Instagram keys
    song   = info.get("music_song")   or info.get("track")  or ""
    artist = info.get("music_artist") or info.get("artist") or ""
    # Fallback: sometimes nested under extras{}
    if not song:
        extras = info.get("extras") or {}
        song   = extras.get("music_song",   "")
        artist = extras.get("music_artist", artist)
    song, artist = str(song).strip(), str(artist).strip()
    if song and artist:
        return f"{song} — {artist}"
    return song or artist


def load_multimodal_assets(prefix: str) -> "pd.DataFrame":
    """Collect transcripts, frame/image GCS URIs, and music info from
    multimodal_dataset_fixed.

    Scans four subtypes:
      feed/     — video: reads frames/ subfolder + transcription.txt + .info.json
      reel/     — video: same structure as feed/
      image/    — photo: reads image files directly in the shortcode folder
      carousel/ — multi-photo: reads all image files in the shortcode folder

    Returns a DataFrame with columns:
      shortcode, subtype, transcript, frame_uris, image_uris, music_info
    """
    fs = _gcsfs.GCSFileSystem()
    bucket_path = prefix.replace("gs://", "")
    records = []

    # --- video subtypes: frames in frames/ subfolder -----------------------
    for subtype in ("feed", "reel"):
        subtype_path = f"{bucket_path}/{subtype}"
        try:
            folders = fs.ls(subtype_path)
        except FileNotFoundError:
            LOGGER.warning("[multimodal] %s/ not found under %s", subtype, prefix)
            continue

        for folder_path in folders:
            shortcode = folder_path.split("/")[-1]

            # transcription
            transcript = ""
            try:
                with fs.open(f"{folder_path}/transcription.txt", "r", encoding="utf-8") as fh:
                    transcript = fh.read().strip()
            except Exception:
                pass

            # frames (cap at 8 to stay within token budget)
            frame_uris = []
            try:
                all_frames = sorted(f for f in fs.ls(f"{folder_path}/frames") if _is_image(f))
                frame_uris = [f"gs://{p}" for p in all_frames[:8]]
            except Exception:
                pass

            # music from .info.json (yt-dlp output)
            music_info = ""
            try:
                info_files = [f for f in fs.ls(folder_path) if f.endswith(".info.json")]
                if info_files:
                    with fs.open(info_files[0], "r", encoding="utf-8") as fh:
                        music_info = _parse_music(_json.load(fh))
            except Exception:
                pass

            records.append({
                "shortcode":   shortcode,
                "subtype":     subtype,
                "transcript":  transcript,
                "frame_uris":  frame_uris,
                "image_uris":  [],
                "music_info":  music_info,
            })

    # --- photo subtypes: images sit directly in shortcode folder -----------
    for subtype in ("image", "carousel"):
        subtype_path = f"{bucket_path}/{subtype}"
        try:
            folders = fs.ls(subtype_path)
        except FileNotFoundError:
            LOGGER.warning("[multimodal] %s/ not found under %s", subtype, prefix)
            continue

        for folder_path in folders:
            shortcode = folder_path.split("/")[-1]
            image_uris = []
            try:
                all_imgs = sorted(f for f in fs.ls(folder_path) if _is_image(f))
                image_uris = [f"gs://{p}" for p in all_imgs[:8]]
            except Exception:
                pass

            records.append({
                "shortcode":   shortcode,
                "subtype":     subtype,
                "transcript":  "",
                "frame_uris":  [],
                "image_uris":  image_uris,
                "music_info":  "",   # photos have no audio track
            })

    df = (
        pd.DataFrame(records)
        if records
        else pd.DataFrame(columns=["shortcode", "subtype", "transcript",
                                    "frame_uris", "image_uris", "music_info"])
    )
    n_frames  = df["frame_uris"].apply(bool).sum()
    n_images  = df["image_uris"].apply(bool).sum()
    n_transc  = df["transcript"].astype(bool).sum()
    n_music   = df["music_info"].astype(bool).sum()
    LOGGER.info(
        "[multimodal] %d assets — %d with frames, %d with images, "
        "%d with transcript, %d with music",
        len(df), n_frames, n_images, n_transc, n_music,
    )
    return df


def load_frames_from_gcs():
    media, comments = {}, {}

    # --- tabular data -------------------------------------------------------
    for plat, fn in MEDIA_FILES.items():
        uri = f"{RAW_PREFIX}/{fn}"
        try:
            media[plat] = _read(uri)
            LOGGER.info("[%s] media: %d rows from %s", plat, len(media[plat]), uri)
        except Exception as exc:
            LOGGER.warning("[%s] media unavailable (%s): %s", plat, uri, exc)

    for plat, fn in COMMENT_FILES.items():
        uri = f"{RAW_PREFIX}/{fn}"
        try:
            comments[plat] = _read(uri)
            LOGGER.info("[%s] comments: %d rows from %s", plat, len(comments[plat]), uri)
        except Exception as exc:
            LOGGER.warning("[%s] comments unavailable (%s): %s", plat, uri, exc)

    yt_parts = []
    for fn in YT_COMMENT_FILES:
        uri = f"{RAW_PREFIX}/{fn}"
        try:
            part = _read(uri)
            yt_parts.append(part)
            LOGGER.info("[youtube] comments shard: %d rows from %s", len(part), uri)
        except Exception as exc:
            LOGGER.warning("[youtube] shard unavailable (%s): %s", uri, exc)
    if yt_parts:
        comments["youtube"] = pd.concat(yt_parts, ignore_index=True)
        LOGGER.info("[youtube] comments total: %d rows", len(comments["youtube"]))

    # --- multimodal asset enrichment (IG only) ------------------------------
    # Joins transcript + frame/image URIs + music_info onto IG posts by shortcode.
    # _frame_uris / _image_uris / _mm_subtype are internal and stripped after
    # Phase 1. music_info is a real column kept through to the output matrix.
    if "instagram" in media:
        try:
            assets = load_multimodal_assets(MULTIMODAL_PREFIX)
            if not assets.empty:
                ig = media["instagram"].copy()
                if "permalink" in ig.columns:
                    ig["_shortcode"] = ig["permalink"].apply(_extract_shortcode)
                    ig = ig.merge(
                        assets.rename(columns={
                            "shortcode":  "_shortcode",
                            "transcript": "_mm_transcript",
                            "subtype":    "_mm_subtype",
                            "frame_uris": "_frame_uris",
                            "image_uris": "_image_uris",
                            # music_info keeps its name — it's a real output column
                        }),
                        on="_shortcode", how="left",
                    )
                    for col in ("_frame_uris", "_image_uris"):
                        ig[col] = ig[col].apply(lambda v: v if isinstance(v, list) else [])
                    ig["music_info"] = ig["music_info"].fillna("")
                    if "transcript" in ig.columns:
                        ig["transcript"] = ig["transcript"].fillna(ig["_mm_transcript"])
                    else:
                        ig["transcript"] = ig["_mm_transcript"]
                    ig = ig.drop(columns=["_shortcode", "_mm_transcript"], errors="ignore")
                    n_vis   = (ig["_frame_uris"].apply(bool) | ig["_image_uris"].apply(bool)).sum()
                    n_music = ig["music_info"].astype(bool).sum()
                    LOGGER.info(
                        "[instagram] multimodal join: %d/%d posts have visuals, %d have music",
                        n_vis, len(ig), n_music,
                    )
                    media["instagram"] = ig
                else:
                    LOGGER.warning("[instagram] no 'permalink' column — multimodal join skipped")
        except Exception as exc:
            LOGGER.warning("[multimodal] asset join failed (non-fatal): %s", exc)

    return media, comments

In [ ]:
# Cell 7 — run (fatal-error guard: log traceback then re-raise so the
# scheduled run is marked FAILED rather than silently "succeeding")
import traceback

try:
    # --- load from GCS -------------------------------------------------------
    media_frames, comment_frames = load_frames_from_gcs()

    if not any(len(d) for d in media_frames.values()):
        LOGGER.warning(
            "No raw media found under %s — using bundled MOCK frames for a smoke run.",
            RAW_PREFIX,
        )
        media_frames, comment_frames = _mock_frames()

    # --- test-mode sampling --------------------------------------------------
    if TEST_MODE:
        for plat in list(media_frames):
            df = media_frames[plat]
            if len(df) > TEST_N_POSTS:
                media_frames[plat] = df.head(TEST_N_POSTS)
                LOGGER.info("TEST MODE: [%s] media capped at %d/%d rows.", plat, TEST_N_POSTS, len(df))
        for plat in list(comment_frames):
            df = comment_frames[plat]
            if len(df) > TEST_N_COMMENTS:
                comment_frames[plat] = df.head(TEST_N_COMMENTS)
                LOGGER.info("TEST MODE: [%s] comments capped at %d/%d rows.", plat, TEST_N_COMMENTS, len(df))

    # --- config (test mode writes to a separate test parquet) ----------------
    _output_uri = (
        f"gs://{GCP_BUCKET}/enriched/test_enriched_post_vibe_matrix.parquet"
        if TEST_MODE else OUTPUT_URI
    )

    config = PipelineConfig(
        gcp_project_id=GCP_PROJECT_ID,
        gcp_location=GCP_LOCATION,
        output_path=_output_uri,
        enable_llm=True,
    )
    pipeline = EnrichmentPipeline(config)
    matrix = pipeline.run(media_frames, comment_frames)
    LOGGER.info(
        "Pipeline finished OK — %d media rows | LLM calls=%d errors=%d → %s",
        len(matrix), pipeline.llm.call_count, pipeline.llm.error_count, _output_uri,
    )
except Exception:
    LOGGER.error("FATAL — pipeline aborted:
%s", traceback.format_exc())
    raise